In [ ]:
#@title Upload BoolNet Rules File
from google.colab import files
import os, textwrap, csv

# 1)  Upload or reuse file
if 'file_rules' in globals() and os.path.exists(file_rules):
    print(f"  Using previously uploaded file: {file_rules}")
else:
    print(textwrap.dedent("""
        Select your rules file (e.g., 'rules_boolnet_XXX.txt').
        You only need to do this ONCE; the path will be stored in the variable 'file_rules'.
    """))
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(" You must upload a rules file.")
    file_rules = list(uploaded.keys())[0]
    print(f"  File uploaded and stored in variable 'file_rules': {file_rules}")

# 2)  Read rules and compute N and <K>
genes   = []        # list of target genes
exprs   = []        # list of expressions (factors)

with open(file_rules, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        tgt = row['targets'].strip()
        fac = row['factors'].strip()
        if tgt:                       # skip empty lines
            genes.append(tgt)
            exprs.append(fac)

N = len(genes)

# → translate each expression to extract tokens
tr_table = str.maketrans({'(': ' ', ')': ' ', '&': ' ', '|': ' ', '!': ' '})
k_list = []
for expr in exprs:
    # remove operators and split into tokens
    tokens = expr.translate(tr_table).split()
    # regulators = genes found in the expression
    regs   = {tok for tok in tokens if tok in genes}
    k_list.append(len(regs))

K_avg = sum(k_list) / N if N else 0.0

print(f"\n  The network contains {N} nodes.")
print(f"  Average K (mean number of regulators per node): {K_avg:.2f}")



Select your rules file (e.g., 'rules_boolnet_XXX.txt').
You only need to do this ONCE; the path will be stored in the variable 'file_rules'.



Saving nsclc_9_nodes.txt to nsclc_9_nodes.txt
  File uploaded and stored in variable 'file_rules': nsclc_9_nodes.txt

  The network contains 9 nodes.
  Average K (mean number of regulators per node): 1.89


In [ ]:
# @title Fitting and validation of rules using real‐time fixed‐point calculation
import itertools
import pandas as pd
import re
import csv
import os
import textwrap
import sys
from google.colab import files

# 0) Load (or reuse) the rules file
if 'file_rules' not in globals() or not os.path.exists(file_rules):
    print(textwrap.dedent("""\
        Upload your rules file (e.g. 'reglas_boolnet_8_nodos.txt').
        It will be stored in the global variable 'file_rules'.
    """))
    upload = files.upload()
    if not upload:
        raise RuntimeError(" You must upload a rules file.")
    file_rules = list(upload.keys())[0]

print(f"  Rules file: {file_rules}")

# 1) Parse BoolNet rules
def load_rules_boolnet(fname: str) -> dict:
    rules = {}
    with open(fname, encoding='utf-8') as fh:
        first = fh.readline().strip()
        fh.seek(0)
        # “Node: expr” format?
        if ":" in first and not first.lower().startswith(("targets", "target")):
            for line in fh:
                if ":" not in line:
                    continue
                node, expr = map(str.strip, line.split(":", 1))
                rules[node] = expr
        else:
            # CSV format
            for row in csv.reader(fh, delimiter=','):
                if not row or len(row) < 2:
                    continue
                node, expr = row[0].strip(), row[1].strip()
                if node.lower().startswith(("targets", "target")):
                    continue
                rules[node] = expr
    return rules

original_rules = load_rules_boolnet(file_rules)
nodes = list(original_rules.keys())
print(" Nodes:", nodes)

# 2) Synchronous simulation functions
def replace_variables(expr: str, values: dict, order: list) -> str:
    for v in order:
        expr = re.sub(rf'\b{re.escape(v)}\b', str(values[v]), expr)
    return expr

def evaluate_network(state, rules):
    ctx = {n: state[i] for i, n in enumerate(nodes)}
    # replace longer names first to avoid partial matches
    order = sorted(nodes, key=len, reverse=True)
    return tuple(
        1 if eval(
            replace_variables(rules[n], ctx, order)
            .replace('!', ' not ')
            .replace('&', ' and ')
            .replace('|', ' or ')
        ) else 0
        for n in nodes
    )

def fixed_points(rules):
    return [
        list(s)
        for s in itertools.product([0, 1], repeat=len(nodes))
        if evaluate_network(s, rules) == s
    ]

fp_list = fixed_points(original_rules)
if not fp_list:
    raise RuntimeError(" No fixed points found.")

print(f" Fixed points found: {len(fp_list)}")
for fp in fp_list:
    print("   ", fp)

df = pd.DataFrame(fp_list, columns=nodes)
desired_attractors = {tuple(a) for a in fp_list}

# 3) Helper functions for rule generation and evaluation
def safe_eval(expr, row):
    order = sorted(df.columns, key=len, reverse=True)
    expr_r = expr
    for col in order:
        expr_r = re.sub(rf'\b{re.escape(col)}\b', f"row['{col}']", expr_r)
    expr_r = expr_r.replace('!', ' not ').replace('&', ' and ').replace('|', ' or ')
    return eval(expr_r)

def generate_expressions(vars_combo):
    exprs = []
    for signs in itertools.product([False, True], repeat=len(vars_combo)):
        terms = [f'!{v}' if s else v for v, s in zip(vars_combo, signs)]
        exprs.extend([" & ".join(terms), " | ".join(terms)])
    return exprs

def generate_and_or_expressions(vars_combo):
    if len(vars_combo) == 2:
        a, b = vars_combo
        out = []
        for s1, s2 in itertools.product([False, True], repeat=2):
            t1 = f'!{a}' if s1 else a
            t2 = f'!{b}' if s2 else b
            out += [f'{t1} & {t2}', f'{t1} | {t2}']
        return out
    if len(vars_combo) == 3:
        a, b, c = vars_combo
        out = []
        for s1, s2, s3 in itertools.product([False, True], repeat=3):
            t1 = f'!{a}' if s1 else a
            t2 = f'!{b}' if s2 else b
            t3 = f'!{c}' if s3 else c
            out += [f'({t1} & {t2}) | {t3}', f'({t1} | {t2}) & {t3}']
        return out
    return []

def candidate_expressions_for_vars(vc):
    exprs = set(generate_expressions(vc))
    exprs.update(generate_and_or_expressions(vc))
    return list(exprs)

def get_rule_components(rule, target):
    return sorted({n for n in nodes if n != target and re.search(rf'\b{re.escape(n)}\b', rule)})

def find_attractors(rules_dict):
    visited, attractors = {}, set()
    for init in itertools.product([0, 1], repeat=len(nodes)):
        path = []
        state = tuple(init)
        while state not in visited:
            visited[state] = None
            path.append(state)
            state = evaluate_network(state, rules_dict)
        if state in path:
            cycle = tuple(path[path.index(state):])
            # record first element for fixed‐point or the full cycle
            attractors.add(cycle[0] if len(cycle) == 1 else cycle)
    return attractors

def compare_with_desired(attractors):
    pf = {a for a in attractors if isinstance(a, tuple) and all(isinstance(x, int) for x in a)}
    return pf == desired_attractors and len(attractors) == len(desired_attractors)

# 4) Main inference process
def main_inference():
    log_lines = []  # to save summary for TXT
    def log(msg):
        print(msg)
        log_lines.append(msg)

    log("\n=== Starting inference process ===")
    default_min = 3
    results = []

    for target in nodes:
        log(f"\nProcessing node {target}…")
        original = original_rules[target]
        inputs = [n for n in nodes if n != target]
        components = get_rule_components(original, target)
        max_comb = max(len(components), default_min)

        total = functional = exact = 0
        valid = []
        seen = set()

        for r in range(1, max_comb + 1):
            for combo in itertools.combinations(inputs, r):
                for cand in candidate_expressions_for_vars(combo):
                    total += 1
                    norm = re.sub(r'[()\s]', '', cand)
                    if (target, norm) in seen:
                        continue
                    seen.add((target, norm))
                    try:
                        res = df.apply(lambda row: int(safe_eval(cand, row)), axis=1)
                        is_func = all(res == df[target])
                    except Exception:
                        is_func = False
                    results.append({"Node": target, "Rule": cand, "Functional": is_func})
                    if is_func:
                        functional += 1
                        tmp_rules = original_rules.copy()
                        tmp_rules[target] = cand
                        if compare_with_desired(find_attractors(tmp_rules)):
                            exact += 1
                            valid.append(cand)

        log(f"   Functional: {functional}")
        log(f"   Exact matches: {exact}")
        log(f"   Original rule: {target} = {original}")
        if valid:
            log("    Valid rules:")
            for v in valid:
                log(f"      {v}")
        else:
            log("    No valid rules found.")

    # Save CSV and TXT
    df_res = pd.DataFrame(results)
    csv_name = "All_rules.csv"
    txt_name = "Summary_inference.txt"
    df_res.to_csv(csv_name, index=False)
    with open(txt_name, "w", encoding="utf-8") as fh:
        fh.write("\n".join(log_lines))

    print(f"\n CSV file: {csv_name}")
    print(f" TXT file: {txt_name}")
    print("=== Process completed ===")
    return csv_name, txt_name

# Run inference
csv_file, txt_file = main_inference()

# Auto-download in Colab (or notification if local)
try:
    files.download(csv_file)
    files.download(txt_file)
except ModuleNotFoundError:
    print(" Generated files:\n", csv_file, "\n", txt_file)


  Rules file: nsclc_9_nodes.txt
 Nodes: ['miR_145', 'Sp1', 'MALAT1', 'BMI1', 'KLF4', 'p53', 'p53_A', 'p53_K', 'E2F1']
 Fixed points found: 3
    [0, 1, 1, 1, 1, 0, 0, 0, 1]
    [1, 0, 0, 0, 0, 1, 0, 1, 0]
    [1, 0, 0, 0, 0, 1, 1, 0, 0]

=== Starting inference process ===

Processing node miR_145…
   Functional: 325
   Exact matches: 0
   Original rule: miR_145 = p53 & !MALAT1 & !BMI1
    No valid rules found.

Processing node Sp1…
   Functional: 325
   Exact matches: 0
   Original rule: Sp1 = (BMI1) | !miR_145
    No valid rules found.

Processing node MALAT1…
   Functional: 325
   Exact matches: 1
   Original rule: MALAT1 = Sp1
    Valid rules:
      (!p53_A & !p53_K) | E2F1

Processing node BMI1…
   Functional: 325
   Exact matches: 2
   Original rule: BMI1 = E2F1
    Valid rules:
      (!p53_A & !p53_K) | E2F1
      (p53_A & p53_K) | E2F1

Processing node KLF4…
   Functional: 325
   Exact matches: 0
   Original rule: KLF4 = !miR_145 | (E2F1 & p53)
    No valid rules found.

Process

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>